# 03 - Streaming de market data

El WebSocket de la v1 entrega puntas, profundidad y operaciones en tiempo real.
`IOLStreamClient` es asincronico: autentica, se suscribe, mantiene el heartbeat
y se reconecta solo.

## Que se ve en este notebook

- Ciclo de vida: `async with` -> `subscribe` -> mensajes -> corte.
- Forma de los mensajes `Q` (puntas), `B` (profundidad) y `T` (operaciones).
- Limites de suscripcion y reparto automatico de conexiones.

**Importante:** el `async for` no termina solo. Siempre hay que cortar con un
tope de mensajes y `break`.


## 1. Credenciales

In [ ]:
import os

# completar: exportar las credenciales en el entorno antes de ejecutar.
USERNAME = os.environ.get("IOL_USERNAME", "")  # completar
PASSWORD = os.environ.get("IOL_PASSWORD", "")  # completar

if not USERNAME or not PASSWORD:
    raise SystemExit("Faltan IOL_USERNAME e IOL_PASSWORD en el entorno")


## 2. Imports y parametros

In [ ]:
import asyncio

from pyIol2 import IOLStreamClient, Markets, SettlementTerms

SYMBOLS = ["GGAL", "YPFD"]
MAX_MENSAJES = 5


## 3. Ciclo de vida

1. **Auth**: el cliente pide el token y manda `auth_token` en cada conexion.
2. **Subscribe**: manda `SIMBOLO|MERCADO|PLAZO` en los canales pedidos.
3. **Heartbeat**: un `ping` cada 15 segundos, porque el servidor no hace ping.
4. **Teardown**: al salir del `async for` o del `async with`, `close()` cancela
   las tareas y cierra las conexiones.


In [ ]:
async def leer_puntas():
    recibidos = 0
    async with IOLStreamClient(USERNAME, PASSWORD) as stream:
        async for mensaje in stream.subscribe(
            SYMBOLS,
            market=Markets.BCBA,
            term=SettlementTerms.T0,
            quotes=True,
            trades=False,
        ):
            print(mensaje.type, mensaje.symbol, mensaje.market, mensaje.term)
            if mensaje.quote is not None:
                for nivel in mensaje.quote.levels:
                    print(" ", nivel.side, nivel.price, nivel.size)
            recibidos += 1
            if recibidos >= MAX_MENSAJES:
                break
    print("mensajes recibidos:", recibidos)


await leer_puntas()


## 4. Operaciones

El canal `trades` trae ultimo precio, variacion y volumen en `StreamTradeMessage`.

In [ ]:
async def leer_operaciones(simbolo):
    recibidos = 0
    async with IOLStreamClient(USERNAME, PASSWORD) as stream:
        async for mensaje in stream.subscribe(
            [simbolo],
            quotes=False,
            trades=True,
        ):
            if mensaje.trade is not None:
                print(
                    mensaje.trade.symbol,
                    mensaje.trade.last_price,
                    mensaje.trade.change_percent,
                    mensaje.trade.traded_quantity,
                )
            recibidos += 1
            if recibidos >= MAX_MENSAJES:
                break


await leer_operaciones("GGAL")


## 5. Profundidad

`order_book=True` admite **un solo simbolo por conexion**. Para puntas y
operaciones el limite es de 50 simbolos por conexion y el reparto es automatico.


In [ ]:
async def leer_profundidad(simbolo):
    recibidos = 0
    async with IOLStreamClient(USERNAME, PASSWORD) as stream:
        async for mensaje in stream.subscribe(
            [simbolo],
            quotes=False,
            trades=False,
            order_book=True,
        ):
            if mensaje.order_book is not None:
                for nivel in mensaje.order_book.levels:
                    print(nivel.side, nivel.position, nivel.price, nivel.size)
            recibidos += 1
            if recibidos >= MAX_MENSAJES:
                break


await leer_profundidad("GGAL")


## 6. Limites y errores

| Caso | Resultado |
|------|-----------|
| Mas de 50 simbolos de puntas u operaciones | Se abren varias conexiones, sin intervencion |
| `order_book` con mas de un simbolo | `IOLValidationError` |
| Lista de simbolos vacia | `IOLValidationError` |
| Sin ningun canal pedido | `IOLValidationError` |
| El servidor rechaza la sesion | `IOLStreamError` con `ws_code` |
| Corte de red | Reconexion con backoff (1s a 30s, 5 intentos) |


In [ ]:
from pyIol2 import IOLValidationError

try:
    async with IOLStreamClient(USERNAME, PASSWORD) as stream:
        async for _ in stream.subscribe(["GGAL", "YPFD"], order_book=True):
            pass
except IOLValidationError as exc:
    print("rechazado antes de conectar:", exc.message)


## 7. Reusar un token

Si ya hay un access token emitido, se puede pasar con `token=` y el cliente no vuelve a pedir credenciales.

In [ ]:
# El token se obtiene con el cliente REST y su TokenManager (ver pyIol2/auth.py).
# stream = IOLStreamClient(token="<access-token>")
print("ver bloque comentado")
